In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

In [2]:
TRAIN_PATH = "./data/mpg_train.csv" # <- change these for new data
TEST_PATH = "./data/mpg_test.csv"
TARGET = "mpg"
ID = "id"
DROP_COLS = ["name"] # other columns that are not features (free text, too many unique values)

In [3]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
train

,id,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,1,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,3,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
2,4,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
3,6,15.0,8,429.0,198.0,4341,10.0,70,usa,ford galaxie 500
4,7,14.0,8,454.0,220.0,4354,9.0,70,usa,chevrolet impala
...,...,...,...,...,...,...,...,...,...,...
313,394,27.0,4,140.0,86.0,2790,15.6,82,usa,ford mustang gl
314,395,44.0,4,97.0,52.0,2130,24.6,82,europe,vw pickup
315,396,32.0,4,135.0,84.0,2295,11.6,82,usa,dodge rampage
316,397,28.0,4,120.0,79.0,2625,18.6,82,usa,ford ranger


In [4]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            318 non-null    int64  
 1   mpg           318 non-null    float64
 2   cylinders     318 non-null    int64  
 3   displacement  318 non-null    float64
 4   horsepower    315 non-null    float64
 5   weight        318 non-null    int64  
 6   acceleration  318 non-null    float64
 7   model_year    318 non-null    int64  
 8   origin        318 non-null    str    
 9   name          318 non-null    str    
dtypes: float64(4), int64(4), str(2)
memory usage: 25.0 KB


In [5]:
train = train.dropna(subset=[TARGET]) # rows without a target can't be used
X = train.drop(columns=[ID, TARGET] + DROP_COLS)
y = train[TARGET]
X_sub = test.drop(columns=[ID] + DROP_COLS) # features of the data to submit

In [6]:
cat_cols = X.select_dtypes(exclude="number").columns # text columns
for c in cat_cols: # same categories for train and test, unseen -> NaN
	cats = X[c].astype("category").cat.categories
	X[c] = pd.Categorical(X[c], categories=cats)
	X_sub[c] = pd.Categorical(X_sub[c], categories=cats)

In [7]:
X_train, X_val, y_train, y_val = train_test_split(
	X, y, test_size=0.2, random_state=42)

In [8]:
model = HistGradientBoostingRegressor(random_state=42) # no scaling, missing values OK
model.fit(X_train, y_train)

,"random_state random_state: int, RandomState instance or None, default=NonePseudo-random number generator to control the subsampling in thebinning process, and the train/validation data split if early stoppingis enabled.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'squared_error', 'absolute_error', 'gamma', 'poisson', 'quantile'}, default='squared_error'The loss function to use in the boosting process. Note that the""squared error"", ""gamma"" and ""poisson"" losses actually implement""half least squares loss"", ""half gamma deviance"" and ""half poissondeviance"" to simplify the computation of the gradient. Furthermore,""gamma"" and ""poisson"" losses internally use a log-link, ""gamma""requires ``y > 0`` and ""poisson"" requires ``y >= 0``.""quantile"" uses the pinball loss... versionchanged:: 0.23 Added option 'poisson'... versionchanged:: 1.1 Added option 'quantile'... versionchanged:: 1.3 Added option 'gamma'.",'squared_error'
,"quantile quantile: float, default=NoneIf loss is ""quantile"", this parameter specifies which quantile to be estimatedand must be between 0 and 1.",None
,"learning_rate learning_rate: float, default=0.1The learning rate, also known as *shrinkage*. This is used as amultiplicative factor for the leaves values. Use ``1`` for noshrinkage.",0.1
,"max_iter max_iter: int, default=100The maximum number of iterations of the boosting process, i.e. themaximum number of trees.",100
,"max_leaf_nodes max_leaf_nodes: int or None, default=31The maximum number of leaves for each tree. Must be strictly greaterthan 1. If None, there is no maximum limit.",31
,"max_depth max_depth: int or None, default=NoneThe maximum depth of each tree. The depth of a tree is the number ofedges to go from the root to the deepest leaf.Depth isn't constrained by default.",None
,"min_samples_leaf min_samples_leaf: int, default=20The minimum number of samples per leaf. For small datasets with lessthan a few hundred samples, it is recommended to lower this valuesince only very shallow trees would be built.",20
,"l2_regularization l2_regularization: float, default=0The L2 regularization parameter penalizing leaves with small hessians.Use ``0`` for no regularization (default).",0.0
,"max_features max_features: float, default=1.0Proportion of randomly chosen features in each and every node split.This is a form of regularization, smaller values make the trees weakerlearners and might prevent overfitting.If interaction constraints from `interaction_cst` are present, only allowedfeatures are taken into account for the subsampling... versionadded:: 1.4",1.0
,"max_bins max_bins: int, default=255The maximum number of bins to use for non-missing values. Beforetraining, each feature of the input array `X` is binned intointeger-valued bins, which allows for a much faster training stage.Features with a small number of unique values may use less than``max_bins`` bins. In addition to the ``max_bins`` bins, one more binis always reserved for missing values. Must be no larger than 255.",255


In [9]:
y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)

print(f"Train  RMSE: {root_mean_squared_error(y_train, y_pred_train):.2f}  R2: {r2_score(y_train, y_pred_train):.3f}")
print(f"Val    RMSE: {root_mean_squared_error(y_val, y_pred_val):.2f}  R2: {r2_score(y_val, y_pred_val):.3f}")

Train  RMSE: 1.73  R2: 0.953
Val    RMSE: 1.90  R2: 0.937


In [10]:
SAVE_CSV = False

submission_df = test[[ID]].copy()
submission_df[TARGET] = model.predict(X_sub)
if SAVE_CSV:
	submission_df.to_csv("submission.csv", index=False)

submission_df

,id,mpg
0,2,13.630063
1,5,17.191874
2,13,14.710733
3,16,19.054500
4,18,22.013163
...,...,...
75,379,37.287036
76,380,34.589676
77,387,23.626252
78,390,24.284179
